In [1]:
import argparse
import pyhocon
from tqdm import tqdm
from itertools import combinations
from sklearn.utils import shuffle
from torch.utils import data

from models import  SpanEmbedder, SpanScorer, FullCrossEncoder
from evaluator import Evaluation
from dataset import CrossEncoderDatasetFull,CrossEncoderMixedDatasetFull
from utils import *
from cluster_documents import return_mentions_with_predicted_topics


C:\Users\sago\AppData\Roaming\Python\Python38\site-packages\transformers\generation_utils.py:24: FutureWarning: Importing `GenerationMixin` from `src/transformers/generation_utils.py` is deprecated and will be removed in Transformers v5. Import as `from transformers import GenerationMixin` instead.
  warnings.warn(
Xformers is not installed correctly. If you want to use memory_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


In [18]:
config = pyhocon.ConfigFactory.parse_file('configs/config_pairwise.json')
fix_seed(config)
logger = create_logger(config, create_file=True)
logger.info(pyhocon.HOCONConverter.convert(config, "hocon"))
create_folder(config.model_path)

In [19]:
config.experiment_id

'Experiment1'

In [3]:
config_experiments = pyhocon.ConfigFactory.parse_file('configs/config_experiments.json')

In [4]:
datasets_config = pyhocon.ConfigFactory.parse_file('configs/config_datasets.json')

datasets_config = pyhocon.ConfigFactory.parse_file('configs/config_datasets.json')
common_data_folder = datasets_config.get_string("common_data_folder")
for key in datasets_config:
    if isinstance(datasets_config[key], dict) and "data_folder" in datasets_config[key]:
        datasets_config[key]["data_folder"] = datasets_config[key]["data_folder"].replace("${common_data_folder}", common_data_folder)

In [20]:
train = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['train'])
val = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['val'])
test = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['test'])

c:\Users\sago\Documents\GitHub\CDCR_lexical_diversity\MTL\cross_encoder_adapted\dataset.py:137: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels,dtype=torch.float)


NUM_TOPICS 5 NUM_MENTIONS 1735


c:\ProgramData\miniconda3\envs\pgkpr\lib\site-packages\sklearn\cluster\_kmeans.py:1366: FutureWarning: algorithm='auto' is deprecated, it will be removed in 1.3. Using 'lloyd' instead.
  warnings.warn(


NUM_TOPICS 5 NUM_MENTIONS 986
NUM_TOPICS 5 NUM_MENTIONS 2622
NUM_TOPICS 5 NUM_MENTIONS 1201


In [9]:
# init train and dev set
#train = CrossEncoderDatasetFull(config, 'train')
#Prepare mentions
#train.create_mention_pairs_by_topics()
#train.negative_sampling()
#train.prepare_pair_of_mentions(mentions=train.mentions, first=train.first, second=train.second)

train_loader = data.DataLoader(train, batch_size=config.batch_size, shuffle=True)

#dev = CrossEncoderDatasetFull(config, 'val')

#Prepare mentions
#predict_topics = True #TODO Pass by config parameter
#if predict_topics:
#    mentions_w_predicted_topics = return_mentions_with_predicted_topics(dev)
#    dev.assign_mentions_with_new_topics(mentions_w_predicted_topics)
#Else by default now: use the known topics at inference, which must change

#dev.create_mention_pairs_by_topics()
#dev.prepare_pair_of_mentions( mentions=dev.mentions, first=dev.first, second=dev.second)

dev_loader = data.DataLoader(val, batch_size=config.batch_size, shuffle=False)

In [16]:
# Explore batches
# Create an iterator for the DataLoader
train_iterator = iter(train_loader)

# Get a batch from the iterator
batch = next(train_iterator)

batch_x, batch_y, instance_id,dataset_task = batch


In [17]:
dataset_task



('CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 'CDCR',
 

In [10]:
mention1[0]
'[SEP]'.join([mention1[0],mention2[0]])

'33_4ecb_HUM17586038555708853_1_25_cEKd[SEP]33_1ecb_Singleton_HUMAyHGGfy_1_9_mq2w'

In [7]:
#Testing document clustering based on Barhom


In [83]:
len(dev.mentions)

3171

In [5]:
new_mentions = return_mentions_with_predicted_topics(dev,compare_with_true_labels=True)

NUM_TOPICS 10 NUM_MENTIONS 3171


/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1366: FutureWarning: algorithm='auto' is deprecated, it will be removed in 1.3. Using 'lloyd' instead.
  warnings.warn(


Homogeneity: 1.000
Completeness: 0.000
V-measure: 0.000
Adjusted Rand-Index: 0.000


In [6]:
device_ids = config.gpu_num
# device = torch.device("cuda:{}".format(device_ids[0]))
device = torch.device("cpu")

In [7]:
config.bert_model

'roberta-large'

In [8]:
device

device(type='cpu')

In [9]:
logger.info('Init models')
span_repr = SpanEmbedder(config, device).to(device)
span_scorer = SpanScorer(config).to(device)
cross_encoder_single = FullCrossEncoder(config).to(device)
#cross_encoder = torch.nn.DataParallel(cross_encoder_single, device_ids=device_ids)

loading file vocab.json from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/vocab.json
loading file merges.txt from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/merges.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at None
loading configuration file config.json from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/config.json
Model config RobertaConfig {
  "_name_or_path": "roberta-large",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 

In [10]:
cross_encoder = cross_encoder_single

In [11]:
if config.training_method in ('pipeline', 'continue') and not config.use_gold_mentions:
    span_repr.load_state_dict(torch.load(config.span_repr_path, map_location=device))
    span_scorer.load_state_dict(torch.load(config.span_scorer_path, map_location=device))



In [12]:
## Optimizer and loss function
criterion = get_loss_function(config)
optimizer = get_optimizer(config, [cross_encoder])
scheduler = get_scheduler(optimizer, total_steps=config.epochs * len(train_loader))



In [18]:
model_params = list(cross_encoder.parameters())

In [25]:
for name, param in cross_encoder.named_parameters():
    print(f"Layer: {name}, Parameters: {param.numel()}")

Layer: model.embeddings.word_embeddings.weight, Parameters: 51473408
Layer: model.embeddings.position_embeddings.weight, Parameters: 526336
Layer: model.embeddings.token_type_embeddings.weight, Parameters: 1024
Layer: model.embeddings.LayerNorm.weight, Parameters: 1024
Layer: model.embeddings.LayerNorm.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.self.query.weight, Parameters: 1048576
Layer: model.encoder.layer.0.attention.self.query.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.self.key.weight, Parameters: 1048576
Layer: model.encoder.layer.0.attention.self.key.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.self.value.weight, Parameters: 1048576
Layer: model.encoder.layer.0.attention.self.value.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.output.dense.weight, Parameters: 1048576
Layer: model.encoder.layer.0.attention.output.dense.bias, Parameters: 1024
Layer: model.encoder.layer.0.attention.output.LayerNorm.weight, Param

In [29]:
config.pretrained_model

AttributeError: 'super' object has no attribute '__getattr__'

In [24]:
model_params[4]

Parameter containing:
tensor([ 0.0315,  0.0422,  0.1929,  ..., -0.2253, -0.0898,  0.1248],
       requires_grad=True)

In [13]:
logger.info('Number of parameters of mention extractor: {}'.format(
    count_parameters(span_repr) + count_parameters(span_scorer)))
logger.info('Number of parameters of the pairwise classifier: {}'.format(
    count_parameters(cross_encoder)))


In [30]:
# Revision prior to training
cross_encoder.train()
# Get one batch
data_batch, labels_batch = next(iter(train_loader))
bert_tokens = cross_encoder.tokenizer(data_batch, pad_to_max_length=True)

/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:2364: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [46]:
cross_encoder.tokenizer.decode(bert_tokens['input_ids'][30])

'<s>Kenneth Dorsey says the [START] woman [END] accused of killing two co - workers and critically injuring a third at the Kraft plant in Northeast Philly is a good person. [SEP] In her statement, [START] Hiller [END] admitted killing two coworkers and seriously wounding a third. [SEP]</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>'

In [42]:
len(bert_tokens['input_ids'][30])

191

In [14]:
##################################################################################
####                    TRAINING
##################################################################################



logger.info('Number of topics: {}'.format(len(train.topics)))
f1 = []
for epoch in range(config.epochs):
    logger.info('Epoch: {}'.format(epoch))
    accumulate_loss = 0
    number_of_positive_pairs, number_of_pairs = 0, 0
    cross_encoder.train()

    for batch_x, batch_y in tqdm(train_loader):
        bert_tokens = cross_encoder.tokenizer(batch_x, pad_to_max_length=True)
        input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
        attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)

        ##Debug

        input_ids = input_ids[:, :5]
        attention_mask = attention_mask[:, :5]
        print("input_ids shape",input_ids.shape)
        print("attention_mask shape",attention_mask.shape)
        
        optimizer.zero_grad()
        cross_encoder.zero_grad()
        scores = cross_encoder(input_ids, attention_mask)
        loss = criterion(scores, batch_y.to(device))
        loss.mean().backward()
        torch.nn.utils.clip_grad_norm_(cross_encoder.parameters(), 1.0)
        optimizer.step()
        scheduler.step()


        accumulate_loss += loss.item()

        number_of_positive_pairs += len((batch_y == 1).nonzero())
        number_of_pairs += len(batch_y)



    # list_of_topics = shuffle(list(range(len(training_set.topic_list))))
    # number_of_positive_pairs = 0
    # number_of_pairs = 0
    # for topic_num in tqdm(list_of_topics):
    #
    #
    #
    #
    #     topic = training_set.topic_list[topic_num]
    #     first, second, labels = get_candidate_spans(training_set, topic)
    #
    #     number_of_positive_pairs += len((labels == 1).nonzero())
    #     number_of_pairs += len(labels)
    #
    #
    #     pairwises = CrossEncoderDataset(training_set.mentions_repr, first, second, labels)
    #     generator = data.DataLoader(pairwises, shuffle=True, batch_size=config.batch_size)
    #     loss = fine_tune_bert(cross_encoder, generator, criterion, optimizer, device)
    #     accumulate_loss += loss


    logger.info('Number of positive/total pairs: {}/{}'.format(number_of_positive_pairs, number_of_pairs))
    logger.info('Accumulate loss: {}'.format(accumulate_loss))


    logger.info('Evaluate on the dev set')
    all_scores, all_labels = [], []
    number_of_positive_pairs, number_of_pairs = 0, 0
    cross_encoder.eval()

    for batch_x, batch_y in tqdm(dev_loader):
        bert_tokens = cross_encoder.tokenizer(batch_x, pad_to_max_length=True)
        input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
        attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)

        with torch.no_grad():
            scores = cross_encoder(input_ids, attention_mask)


        number_of_positive_pairs += len((batch_y == 1).nonzero())
        number_of_pairs += len(batch_y)


    # for topic_num, topic in enumerate(tqdm(dev_set.topic_list)):
    #     dev_topic = dev_set.topic_list[topic_num]
    #     first, second, labels = get_candidate_spans(dev_set, dev_topic)
    #     number_of_positive_pairs += len((labels == 1).nonzero())
    #     number_of_pairs += len(labels)
    #
    #     pairwises = CrossEncoderDataset(dev_set.mentions_repr, first, second, labels)
    #     generator = data.DataLoader(pairwises, shuffle=True, batch_size=config.batch_size)
    #     scores, labels = evaluate_model(cross_encoder, generator, device)
    #     torch.cuda.empty_cache()
    #
        all_scores.extend(scores.squeeze(1))
        all_labels.extend(batch_y.squeeze(1))



    all_labels = torch.stack(all_labels)
    all_scores = torch.stack(all_scores)


    strict_preds = (all_scores > 0).to(torch.int)
    eval = Evaluation(strict_preds, all_labels.to(device))
    logger.info('Number of predictions: {}/{}'.format(strict_preds.sum(), len(strict_preds)))
    logger.info('Number of positive pairs: {}/{}'.format(len((all_labels == 1).nonzero()),
                                                            len(all_labels)))

    logger.info('Min score: {}'.format(all_scores.min().item()))
    logger.info('Max score: {}'.format(all_scores.max().item()))
    logger.info('Strict - Recall: {}, Precision: {}, F1: {}, Accuracy: {}'.
                format(eval.get_recall(), eval.get_precision(), eval.get_f1(), eval.get_accuracy()))


    out_dir = os.path.join(config.model_path, '{}_{}'.format(
        'base' if 'base' in config.roberta_model else 'large', config.batch_size),
        'checkpoint_{}'.format(epoch))
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)


    cross_encoder_single.model.save_pretrained(os.path.join(out_dir, 'bert'))
    torch.save(cross_encoder_single.linear.state_dict(), os.path.join(out_dir, 'linear'))

    f1.append(eval.get_f1())


  0%|          | 0/7527 [00:00<?, ?it/s]

/home/sago/miniconda3/envs/pgkpr/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:2364: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


input_ids shape torch.Size([32, 5])
attention_mask shape torch.Size([32, 5])


  0%|          | 1/7527 [00:05<10:49:29,  5.18s/it]

input_ids shape torch.Size([32, 5])
attention_mask shape torch.Size([32, 5])


...

input_ids shape torch.Size([32, 5])
attention_mask shape torch.Size([32, 5])


  0%|          | 10/7527 [00:45<9:18:46,  4.46s/it]

input_ids shape torch.Size([32, 5])
attention_mask shape torch.Size([32, 5])


  0%|          | 10/7527 [00:49<10:15:38,  4.91s/it]


KeyboardInterrupt: 

In [ ]:
explore_scores.last_hidden_state[:, 0, :].shape

torch.Size([32, 1024])